This notebook tests the FSRCNN model using a **given ground-truth image** by creating its low-resolution
counterpart and applying super-resolution on it.

In [10]:
from pathlib import Path
import os

import numpy as np
import torch
from PIL import Image
import onnxruntime as ort
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

Adjust this part based on current project structure.

In [11]:
image_path = r""
output_path = r''
DOWNSCALE_FACTOR = 4
TARGET_WIDTH, TARGET_HEIGHT = 178 // DOWNSCALE_FACTOR, 218 // DOWNSCALE_FACTOR
TARGET_RATIO = 3 / 4

In [12]:
with Image.open(image_path) as image:
    source_width, source_height = image.size
    source_ratio = source_width / source_height

    if source_ratio > TARGET_RATIO:
        # Image is wider than 3:4, crop left/right
        new_width = int(source_height * TARGET_RATIO)
        new_height = source_height
    else:
        # Image is taller than 3:4, crop top/bottom
        new_width = source_width
        new_height = int(source_width / source_ratio)

    left = (source_width - new_width) // 2
    top = (source_height - new_height) // 2
    right = left + new_width
    bottom = top  + new_height

    # Crop image
    image_cropped = image.crop((left, top, right, bottom))

    # Resize to desired weight and height using bicubic interpolation
    # Use downscaled width and height to resize by a downscale factor
    # Alternatively, pass TARGET_WIDTH and TARGET_HEIGHT to resize to a specific dimension
    downscaled_width, downscaled_height = int(new_width / DOWNSCALE_FACTOR), int(new_height / DOWNSCALE_FACTOR)
    image_resized = image_cropped.resize((TARGET_WIDTH, TARGET_HEIGHT), Image.BICUBIC)

    # Save image
    if not output_path:
        base, ext = os.path.splitext(image_path)
        output_path = f"{base}_cropped{ext}"

    image_resized.save(output_path)
    print(f"Saved image at {output_path} (size {image_resized.size})")

Saved image at C:\Users\Raul\Pictures\Camera Roll\WhatsApp Image 2025-03-18 at 09.56.21_7cf0916b_cropped.jpg (size (300, 400))
